In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://www.vietjack.com/trac-nghiem-dai-hoc/trac-nghiem-luat-cong-chung.jsp"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Lấy text từ phần nội dung chính
content = soup.find('div', class_='content') or soup.body
text = content.get_text("\n", strip=True)

# Tách từng câu hỏi (chấp nhận xuống dòng giữa số câu và dấu chấm)
question_pattern = r"(Câu\s+\d+\s*[\.:].*?)(?=Câu\s+\d+\s*[\.:]|TRẮC NGHIỆM ONLINE|$)"
question_blocks = re.findall(question_pattern, text, flags=re.DOTALL)

questions = []
for block in question_blocks:
    q_text_match = re.search(r"(Câu\s+\d+\s*[\.:].*?)(?=\nA\.|Hiển thị đáp án|$)", block, flags=re.DOTALL)
    if q_text_match:
        full_q = q_text_match.group(1)
        # Xóa tiền tố 'Câu X :'
        clean_q = re.sub(r"^Câu\s+\d+\s*[\.:]\s*", "", full_q, flags=re.DOTALL).strip()
        # Chuẩn hóa khoảng trắng
        clean_q = re.sub(r"\s+", " ", clean_q)

        if clean_q:
            questions.append(clean_q)

df = pd.DataFrame(questions, columns=["question"])

if not df.empty:
    print(df.head())
    df.to_csv("luat_cong_chung_questions.csv", index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu {len(df)} câu hỏi vào file luat_cong_chung_questions.csv")
else:
    print("Không tìm thấy câu hỏi nào. Vui lòng kiểm tra lại cấu trúc trang web.")

                                            question
0  Theo quy định của pháp luật hiện hành, khi thự...
1  Theo quy định của pháp luật hiện hành, việc cô...
2  Theo quy định của pháp luật hiện hành, công ch...
3  Theo quy định của pháp luật hiện hành, việc cô...
4  Theo quy định của pháp luật hiện hành, tổ chức...

Đã lưu 20 câu hỏi vào file luat_cong_chung_questions.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json

url = "https://baitaptracnghiem.com/lam-bai/thi-thu-trac-nghiem-on-tap-luat-hinh-su-online-de-11"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# Lấy toàn bộ text
text = soup.get_text("\n", strip=True)

lines = [line.strip() for line in text.split("\n") if line.strip()]

questions_data = []

current_question = None
current_answers = []

# Regex nhận diện câu hỏi
question_pattern = re.compile(r"^Câu\s+\d+[:.]?$")

# Regex đáp án A B C D
answer_pattern = re.compile(r"^[A-D]\.\s*(.+)")

i = 0

while i < len(lines):

    line = lines[i]

    # Nếu gặp Câu x
    if question_pattern.match(line):

        # Lưu câu trước đó
        if current_question and len(current_answers) > 0:
            questions_data.append({
                "question": current_question,
                "answers": current_answers,
                "correct_answer": None
            })

        current_question = None
        current_answers = []

        # Dòng kế tiếp là nội dung câu hỏi
        if i + 1 < len(lines):
            current_question = lines[i + 1]
            i += 1

    else:
        # Kiểm tra đáp án
        match = answer_pattern.match(line)

        if match:
            answer_text = match.group(1).strip()
            current_answers.append(answer_text)

    i += 1

# Lưu câu cuối
if current_question and len(current_answers) > 0:
    questions_data.append({
        "question": current_question,
        "answers": current_answers,
        "correct_answer": None
    })


# Chuyển answers thành JSON string
for item in questions_data:
    item["answers"] = json.dumps(item["answers"], ensure_ascii=False)

# DataFrame
df = pd.DataFrame(questions_data)

# Xuất CSV
df.to_csv("luat_hinh_su_de_11.csv", index=False, encoding="utf-8-sig")

print(df.head())

print("\nĐã lưu file: luat_hinh_su_de_11.csv")

                                            question  \
0         Thời hạn tạm giam của tội nghiêm trọng là:   
1                 Thẩm quyền cấm đi khỏi nơi cư trú:   
2  Nhà nước xây dựng quân đội nhân dân trên cơ sở...   
3  Kì hộp Quốc hội thứ I Khoá mới được triệu tập ...   
4  Chương trình hoạt động của Hội đồng dân tộc và...   

                                             answers correct_answer  
0  ["Không quá 3 tháng.", "2 tháng.", "Không quá ...           None  
1  ["Chủ tịch UBND xã.", "Chủ tịch UBND huyện.", ...           None  
2  ["Kết hợp xây dựng với bảo vệ tổ quốc", "Kết h...           None  
3   ["3 tháng.", "2 tháng.", "1 tháng.", "15 ngày."]           None  
4  ["Quốc hội quyết định.", "UBTV Quốc hội quyết ...           None  

Đã lưu file: luat_hinh_su_de_11.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re

url = "https://thuviencntt.com/cau-hoi-trac-nghiem-ve-an-ninh-mang/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

soup = BeautifulSoup(html, "html.parser")

text = soup.get_text("\n", strip=True)

lines = [
    line.strip()
    for line in text.split("\n")
    if line.strip()
]

questions_data = []

current_question = None
current_answers = []

question_pattern = re.compile(
    r"^(Câu|Question)\s*\d+[:.]?",
    re.IGNORECASE
)

answer_pattern = re.compile(
    r"^[A-D][\.\)]\s*(.+)"
)

for line in lines:

    # Câu hỏi
    if question_pattern.match(line):

        # lưu câu cũ
        if current_question and current_answers:
            questions_data.append({
                "question": current_question,
                "answers": current_answers,
                "correct_answer": None
            })

        current_answers = []

        current_question = re.sub(
            r"^(Câu|Question)\s*\d+[:.]?\s*",
            "",
            line,
            flags=re.IGNORECASE
        ).strip()

    else:
        # đáp án
        m = answer_pattern.match(line)

        if m:
            answer = m.group(1).strip()
            current_answers.append(answer)

# lưu câu cuối
if current_question and current_answers:
    questions_data.append({
        "question": current_question,
        "answers": current_answers,
        "correct_answer": None
    })

# answers -> json string
for item in questions_data:
    item["answers"] = json.dumps(
        item["answers"],
        ensure_ascii=False
    )

df = pd.DataFrame(questions_data)

df.to_csv(
    "an_ninh_mang_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head(10))
print("\nTổng số câu:", len(df))

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json

url = "https://tracnghiemcongchuc.com/luat-luu-tru-2024/39-cau-hoi-trac-nghiem-luat-luu-tru-2024-so-1-free-1789.html"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# Lấy toàn bộ text
text = soup.get_text("\n", strip=True)

lines = [line.strip() for line in text.split("\n") if line.strip()]

questions_data = []

current_question = None
current_answers = []

# Regex nhận diện câu hỏi
question_pattern = re.compile(r"^Câu\s+\d+[:.]?$")

# Regex đáp án A B C D
answer_pattern = re.compile(r"^[A-D]\.\s*(.+)")

i = 0

while i < len(lines):

    line = lines[i]

    # Nếu gặp Câu x
    if question_pattern.match(line):

        # Lưu câu trước đó
        if current_question and len(current_answers) > 0:
            questions_data.append({
                "question": current_question,
                "answers": current_answers,
                "correct_answer": None
            })

        current_question = None
        current_answers = []

        # Dòng kế tiếp là nội dung câu hỏi
        if i + 1 < len(lines):
            current_question = lines[i + 1]
            i += 1

    else:
        # Kiểm tra đáp án
        match = answer_pattern.match(line)

        if match:
            answer_text = match.group(1).strip()
            current_answers.append(answer_text)

    i += 1

# Lưu câu cuối
if current_question and len(current_answers) > 0:
    questions_data.append({
        "question": current_question,
        "answers": current_answers,
        "correct_answer": None
    })


# Chuyển answers thành JSON string
for item in questions_data:
    item["answers"] = json.dumps(item["answers"], ensure_ascii=False)

# DataFrame
df = pd.DataFrame(questions_data)

# Xuất CSV
df.to_csv("luat_dan_su_de_1.csv", index=False, encoding="utf-8-sig")

print(df.head())

print("\nĐã lưu file: luat_dan_su_de_1.csv")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re

url = "https://thuvienphapluat.vn/banan/tin-tuc/cau-hoi-va-dap-an-cuoc-thi-truc-tuyen-tim-hieu-luat-an-ninh-mang-tinh-soc-trang-12404?rel=ban_an_chitietvb"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# Lấy toàn bộ text
text = soup.get_text("\n")

# Tách dòng và làm sạch
lines = [
    line.strip()
    for line in text.split("\n")
    if line.strip()
]

questions_data = []

current_question = None
current_answers = []

# Regex
question_pattern = re.compile(r"^Câu\s+\d+")
answer_pattern = re.compile(r"^[A-D][\.\)]\s*(.+)")

for line in lines:
    # Nếu là câu hỏi
    if question_pattern.match(line):

        # Lưu câu cũ
        if current_question and len(current_answers) > 0:
            questions_data.append({
                "question": current_question,
                "answers": current_answers,
                "correct_answer": None
            })

        current_answers = []

        # Bỏ phần "Câu x ..."
        current_question = re.sub(
            r"^Câu\s+\d+\s*(\([^)]+\))?[:.]?\s*",
            "",
            line
        ).strip()

    else:
        # Nếu là đáp án
        match = answer_pattern.match(line)

        if match:
            answer_text = match.group(1).strip()

            current_answers.append(answer_text)

# lưu câu cuối
if current_question and len(current_answers) > 0:
    questions_data.append({
        "question": current_question,
        "answers": current_answers,
        "correct_answer": None
    })

# Convert answers thành JSON string
for item in questions_data:
    item["answers"] = json.dumps(
        item["answers"],
        ensure_ascii=False
    )

# DataFrame
df = pd.DataFrame(questions_data)

# Xuất CSV
df.to_csv(
    "soc_trang_an_ninh_mang.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head(10))

print("\nTổng số câu hỏi:", len(df))
print("Đã lưu file soc_trang_an_ninh_mang.csv")

Empty DataFrame
Columns: []
Index: []

Tổng số câu hỏi: 0
Đã lưu file soc_trang_an_ninh_mang.csv
